Input Acquisition, takes the invoice image path and put it in an input file

*   List item
*   List item

inside the OCR Project File

In [ ]:
import os
import uuid
from PIL import Image
import shutil
import logging
import glob

# Allowed image formats
VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".tiff"]

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("image_handler.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def validate_image_format(filepath):
    """
    Validates if the file extension is in allowed image formats.
    """
    ext = os.path.splitext(filepath)[1].lower()
    return ext in VALID_EXTENSIONS

def load_image(filepath):
    """
    Opens and returns the image using Pillow.
    """
    if not validate_image_format(filepath):
        raise ValueError("Unsupported image format.")
    return Image.open(filepath)

def save_uploaded_image(source_path, dest_folder=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input"):
    """
    Saves uploaded image with a UUID name to the input directory.

    Args:
        source_path (str): Path to the image file being uploaded.
        dest_folder (str): Folder to store the image in.

    Returns:
        str: Full path where the image is stored.
    """
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)

    if not validate_image_format(source_path):
        raise ValueError("Unsupported image format.")

    ext = os.path.splitext(source_path)[1].lower()
    unique_filename = f"{uuid.uuid4()}{ext}"
    dest_path = os.path.join(dest_folder, unique_filename)

    shutil.copy2(source_path, dest_path)
    return dest_path

def process_images(input_path):
    """
    Process a single image or all images in a directory.

    Args:
        input_path (str): Path to a single image or directory containing images.
    """
    if os.path.isdir(input_path):
        image_files = glob.glob(os.path.join(input_path, f"*{VALID_EXTENSIONS[0]}")) + \
                     glob.glob(os.path.join(input_path, f"*{VALID_EXTENSIONS[1]}")) + \
                     glob.glob(os.path.join(input_path, f"*{VALID_EXTENSIONS[2]}")) + \
                     glob.glob(os.path.join(input_path, f"*{VALID_EXTENSIONS[3]}"))
        if not image_files:
            logger.error(f"No valid image files found in {input_path}")
            return
    elif os.path.isfile(input_path):
        if not validate_image_format(input_path):
            logger.error(f"Unsupported image format for {input_path}")
            return
        image_files = [input_path]
    else:
        logger.error(f"Invalid path: {input_path} is neither a file nor a directory")
        return

    for image_file in image_files:
        try:
            logger.info(f"Processing image: {image_file}")
            load_image(image_file)  # Validate by loading
            dest_path = save_uploaded_image(image_file)
            logger.info(f"Image saved successfully to {dest_path}")
        except Exception as e:
            logger.error(f"Failed to process {image_file}: {str(e)}")

def main():
    """Main program to handle image processing."""
    input_dir = r"C:\Users\dell\Downloads\Screenshot 2025-09-10 133133.jpg"
    logger.info(f"Starting image processing from {input_dir}")
    process_images(input_dir)
    logger.info("Image processing completed")

if __name__ == "__main__":
    main()

2025-09-10 13:32:49,950 - INFO - Starting image processing from C:\Users\dell\Downloads\Screenshot 2025-09-10 133133.jpg
2025-09-10 13:32:49,952 - INFO - Processing image: C:\Users\dell\Downloads\Screenshot 2025-09-10 133133.jpg
2025-09-10 13:32:49,956 - INFO - Image saved successfully to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input\b0be2d26-cd9a-4280-a8e4-9740b9bee7b0.jpg
2025-09-10 13:32:49,957 - INFO - Image processing completed


OCR Data Extraction Using Qwen2.5vl:72B

---



In [ ]:
import os
import glob
import logging
import base64
import io
import re
import time
from openai import OpenAI
from dotenv import load_dotenv
from PIL import Image

# Set up logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("document_extraction.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv(r"C:\Users\dell\OneDrive\Documents\invoice_ocr_pipeline\.env")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("QWEN_API_KEY"),
)

def load_image(image_path):
    """Load and compress the invoice image for Qwen-VL."""
    logger.info(f"Loading and compressing image from {image_path}")
    try:
        img = Image.open(image_path)
        img.thumbnail((800, 800), Image.Resampling.LANCZOS)
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG", quality=85)
        image_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return image_base64
    except Exception as e:
        logger.error(f"Error loading or compressing image: {e}")
        raise

def extract_raw_text(image_base64, max_retries=3):
    """Extract raw text from the invoice image using Qwen-VL."""
    logger.info("Extracting raw text from Qwen-VL")
    prompt = (
        "Analyze this invoice image and extract all text. Return the text in a plain format, including all detected content "
        "from regions, lines, and tables, without requiring a specific JSON structure. Ensure all text is captured, "
        "even in complex layouts, and separate lines with newlines."
    )
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                extra_headers={
                    "HTTP-Referer": "http://localhost",
                    "X-Title": "Document Extraction"
                },
                model="qwen/qwen2.5-vl-72b-instruct:free",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                        ]
                    }
                ],
                max_tokens=16384
            )
            raw_response = completion.choices[0].message.content
            logger.info(f"Raw Qwen-VL response (first 500 chars): {raw_response[:500]}...")
            with open("raw_text.log", "w", encoding="utf-8") as f:
                f.write(raw_response)
            # Extract plain text, removing any JSON-like artifacts
            cleaned_text = re.sub(r'^``(?:json)?\s*|\s*``$', '', raw_response, flags=re.MULTILINE).strip()
            if not cleaned_text:
                logger.warning("No text extracted, returning empty string")
                return ""
            return cleaned_text
        except Exception as e:
            logger.error(f"Qwen-VL API call failed: {str(e)}")
            if "429" in str(e):
                reset_time = int(e.response.headers.get("X-RateLimit-Reset", time.time() + 60))
                wait_time = max(1, reset_time - time.time())
                logger.info(f"Rate limit hit, waiting {wait_time} seconds before retry")
                time.sleep(wait_time)
            elif attempt < max_retries - 1:
                logger.info(f"Retrying API call in 2 seconds due to error")
                time.sleep(2)
            else:
                logger.warning("Max retries reached, no text extracted")
                return ""
    return ""

def save_processed_data(raw_text, newest_file, output_dir):
    """Save the extracted raw text to a text file for post-processing."""
    logger.info(f"Preparing to save raw text to {output_dir}")
    try:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(newest_file)
        base_name = os.path.splitext(base_name)[0] + "_raw.txt"
        text_output_path = os.path.join(output_dir, base_name)

        with open(text_output_path, 'w', encoding='utf-8') as f:
            f.write(raw_text)

        logger.info(f"Raw text saved to {text_output_path}")
        return text_output_path
    except Exception as e:
        logger.error(f"Error saving raw text: {e}")
        raise


def extract_document(image_path, output_dir):
    """Orchestrate the document extraction process using Qwen-VL."""
    logger.info(f"Starting document extraction for {image_path}")
    image_base64 = load_image(image_path)
    raw_text = extract_raw_text(image_base64)

    if not raw_text:
        logger.warning("No text extracted, using empty file")
        raw_text = ""

    text_output_path = save_processed_data(raw_text, image_path, output_dir)
    return text_output_path


def process_documents(input_dir, output_dir):
    """
    Process all images in the input directory or the latest image.
    """
    if not os.path.exists(input_dir):
        logger.error(f"Input directory {input_dir} does not exist")
        return

    image_files = glob.glob(os.path.join(input_dir, "*.jpg")) + \
                 glob.glob(os.path.join(input_dir, "*.jpeg")) + \
                 glob.glob(os.path.join(input_dir, "*.png")) + \
                 glob.glob(os.path.join(input_dir, "*.bmp"))

    if not image_files:
        logger.error(f"No valid image files found in {input_dir}")
        return

    for image_file in image_files:
        text_output_path = extract_document(image_file, output_dir)
        logger.info(f"Document extraction for {image_file} completed, raw text saved to {text_output_path}")


def main():
    """Main program to handle document extraction."""
    input_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input"
    output_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed"

    logger.info(f"Starting document extraction from {input_dir} to {output_dir}")
    process_documents(input_dir, output_dir)
    logger.info("Document extraction completed")


if __name__ == "__main__":
    main()


2025-09-10 13:33:25,359 - INFO - Starting document extraction from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed
2025-09-10 13:33:25,368 - INFO - Starting document extraction for C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input\9551f2fd-8103-489e-9c6c-e9aefe035ad9.jpg
2025-09-10 13:33:25,369 - INFO - Loading and compressing image from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\1. input\9551f2fd-8103-489e-9c6c-e9aefe035ad9.jpg
2025-09-10 13:33:25,379 - INFO - Extracting raw text from Qwen-VL
2025-09-10 13:33:27,762 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\dell\miniconda3\envs\ocr_invoice_pipeline\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\dell\miniconda3\envs\ocr_invoice_pipeline\lib\

In [ ]:
Post Processing

In [ ]:
import os
import glob
import logging
import re
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("text_post_processing.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def process_raw_text(text_path):
    """Process raw text from the extracted file into a cleaned format."""
    logger.info(f"Processing raw text from {text_path}")
    try:
        with open(text_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        # Clean and normalize the text
        lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
        cleaned_text = '\n'.join(lines)

        # Optional: Add basic formatting (e.g., grouping related lines)
        processed_text = cleaned_text
        return processed_text
    except Exception as e:
        logger.error(f"Error processing raw text: {e}")
        return ""

def save_processed_text(processed_text, input_path, output_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed"):
    """Save the processed text to a text file."""
    logger.info(f"Preparing to save processed text to {output_dir}")
    try:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(input_path).replace("_raw.txt", "_processed.txt") if input_path.endswith("_raw.txt") else f"{os.path.splitext(input_path)[0]}_processed.txt"
        text_output_path = os.path.join(output_dir, base_name)
        with open(text_output_path, 'w', encoding='utf-8') as f:
            f.write(processed_text)
        logger.info(f"Processed text saved to {text_output_path}")
        return text_output_path
    except Exception as e:
        logger.error(f"Error saving processed text: {e}")
        raise

def post_process_invoice(text_path):
    """Orchestrate the post-processing of extracted raw text."""
    logger.info(f"Starting post-processing for {text_path}")
    processed_text = process_raw_text(text_path)
    if not processed_text:
        logger.warning("No processed text to save, using empty file")
        processed_text = ""
    text_output_path = save_processed_text(processed_text, text_path)
    logger.info(f"Post-processing completed, text saved to {text_output_path}")
    return text_output_path

def process_all_texts(input_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed"):
    """
    Process all raw text files in the input directory.
    """
    if not os.path.exists(input_dir):
        logger.error(f"Input directory {input_dir} does not exist")
        return
    text_files = glob.glob(os.path.join(input_dir, "*_raw.txt"))
    if not text_files:
        logger.error(f"No raw text files found in {input_dir}")
        return
    for text_file in text_files:
        text_output_path = post_process_invoice(text_file)
        logger.info(f"Post-processing for {text_file} completed, text saved to {text_output_path}")

def main():
    """Main program to handle text post-processing."""
    input_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed"
    logger.info(f"Starting text post-processing from {input_dir}")
    process_all_texts(input_dir)
    logger.info("Text post-processing completed")

if __name__ == "__main__":
    main()

2025-09-10 15:52:39,426 - INFO - Starting text post-processing from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed
2025-09-10 15:52:39,439 - INFO - Starting post-processing for C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9_raw.txt
2025-09-10 15:52:39,439 - INFO - Processing raw text from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\3. processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9_raw.txt
2025-09-10 15:52:39,446 - INFO - Preparing to save processed text to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed
2025-09-10 15:52:39,448 - INFO - Processed text saved to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9_processed.txt
2025-09-10 15:52:39,448 - INFO - Post-processing completed, text saved to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9

Key Value Pair Extraction , this Expected Output Format example in the prompt is for general purposes, if you want to follow a specific invoice structure, change it for better accuracy.

In [ ]:
import os
import glob
import logging
import json
from openai import OpenAI

# Set up logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("invoice_field_mapping.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# === API Configuration ===
API_KEY = "sk-or-v1-REDACTED"  # Your OpenRouter API key
API_BASE = "https://openrouter.ai/api/v1"  # ✅ OpenRouter base URL
SITE_URL = "https://your-site.com"  # ✅ Replace with your actual site
SITE_NAME = "InvoiceExtractorApp"  # ✅ Your app name

def load_text(text_path):
    """Load the processed text file."""
    with open(text_path, 'r', encoding='utf-8') as f:
        return f.read()

def call_qwen_vl(text_content, fallback_txt_path=None):
    """Call Qwen-VL via OpenRouter API and return structured result or raw text."""
    client = OpenAI(
        base_url=API_BASE,
        api_key=API_KEY,
    )

    structured_prompt = """
    ### Role
You are an expert data extractor and OCR correction specialist, specialized in processing invoice text from any language (e.g., French, English). You understand invoice structures, line items, amounts, quantities, dates, and additional info. You preserve the original language of the invoice and the exact wording of keys and values.

### Context
You are provided with raw text extracted from an invoice. The invoice layout, language, and field names may vary. The text may include headers, additional information, line items, totals, and other invoice-specific data. OCR errors may be present, such as:
- '{', '[', '\u00e8' instead of 'é'
- Misplaced '@' characters
- Other non-alphanumeric characters (.,#,$, etc.) that distort meaning

Line items may include columns like position, item number, description, quantity, unit price, amount, and tax rate, but the exact names may vary. Quantities, amounts, and prices may require unit symbols.

### Task
1. Extract all invoice data into a structured **text report** with the following sections:
   - `=== Invoice Header ===`
   - `=== Additional Information ===`
   - `=== Line Items ===`
2. Correct OCR errors:
   - Replace '{', '[', '\u00e8' → 'é'
   - Replace misplaced '@' with context-appropriate character or remove
   - Replace '\n' with empty spaces ' '
3. Remove or replace other non-alphanumeric characters that distort meaning.
4. In **Invoice Header**, always include:
   - Invoice Id
   - Invoice Date (convert to YYYY-MM-DD)
   - Issuer
   - Recipient
   - Total (with currency if available)
5. In **Additional Information**, list all other key-value pairs detected in the invoice (dates, client IDs, terms, delivery conditions, etc).
6. In **Line Items**:
   - Always number them with `Position: X`
   - Include Item Number, Description, Quantity (with unit if available, add the unit besides the quantity number if it exist), Unit Price (with currency, if not available use the currency of the issuer country), Amount (with currency, if not available use the currency of the issuer country), and Tax Rate (%).
   - Keep numbers as they are with commas and full decimals.
   - Use "N/A" for missing strings and "0.0" for missing numbers.
7. Detect the invoice language and preserve the **original keys and language** in Additional Information and Line Items.
8. Respond **ONLY with the formatted text output**, no explanations or markdown.
9. If extraction fails, output: `Unable to validate invoice data`.

### Expected Output Format
=== Invoice Header ===
Invoice Id: ...
Invoice Date: ...
Issuer: ...
Recipient: ...
Total: ...

=== Additional Information ===
Key1: Value1
Key2: Value2
...

=== Line Items ===
Position: 1
Item Number: ...
Description: ...
Quantity: ...
Unit Price: ...
Amount: ...
Tax Rate: ...

Position: 2
...

    """

    try:
        response = client.chat.completions.create(
            model="qwen/qwen2.5-vl-72b-instruct:free",
            messages=[
                {"role": "system", "content": structured_prompt},
                {"role": "user", "content": text_content}
            ],
            max_tokens=4096,
            temperature=0.2,
            extra_headers={
                "HTTP-Referer": SITE_URL,
                "X-Title": SITE_NAME,
            },
        )

        # ✅ Just treat it as raw structured text (not JSON)
        raw_content = response.choices[0].message.content.strip()
        logger.info(f"Raw model response:\n{raw_content}")
        return raw_content

    except Exception as e:
        logger.error(f"Qwen-VL API error: {str(e)}")
        return None


def extract_key_value_pairs(text_path):
    """Extract key-value pairs using Qwen-VL from processed text."""
    logger.info(f"Extracting key-value pairs from text {text_path}")
    try:
        text_content = load_text(text_path)
        qwen_response = call_qwen_vl(text_content)

        # ✅ Qwen returns structured text already → pass it forward
        if qwen_response:
            return qwen_response
        else:
            logger.warning("Qwen-VL returned no valid data, falling back to basic parsing")
            with open(text_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
            return raw_text if raw_text.strip() else ""
    except Exception as e:
        logger.error(f"Error extracting key-value pairs: {str(e)}")
        return ""

def save_mapped_data(mapped_text, input_path, output_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped"):
    """Save the mapped key-value pairs to a text file."""
    logger.info(f"Preparing to save mapped data to {output_dir}")
    try:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(input_path).replace("_processed.txt", "_mapped.txt")
        mapped_output_path = os.path.join(output_dir, base_name)
        with open(mapped_output_path, 'w', encoding='utf-8') as f:
            f.write(mapped_text)
        logger.info(f"Mapped data saved to {mapped_output_path}")
        return mapped_output_path
    except Exception as e:
        logger.error(f"Error saving mapped data: {e}")
        raise

def map_invoice_fields(text_path):
    """Orchestrate the field mapping process using Qwen-VL."""
    logger.info(f"Starting field mapping for {text_path}")
    mapped_text = extract_key_value_pairs(text_path)
    if not mapped_text:
        logger.warning("No mapped data to save, using empty file")
        mapped_text = ""
    mapped_output_path = save_mapped_data(mapped_text, text_path)
    logger.info(f"Field mapping completed, data saved to {mapped_output_path}")
    return mapped_output_path

def process_all_files(input_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed"):
    """
    Process all processed text files in the input directory.
    """
    if not os.path.exists(input_dir):
        logger.error(f"Input directory {input_dir} does not exist")
        return
    text_files = glob.glob(os.path.join(input_dir, "*_processed.txt"))
    if not text_files:
        logger.error(f"No processed text files found in {input_dir}")
        return
    for text_file in text_files:
        mapped_output_path = map_invoice_fields(text_file)
        logger.info(f"Field mapping for {text_file} completed, data saved to {mapped_output_path}")

def main():
    """Main program to handle invoice field mapping."""
    input_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed"
    output_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped"
    logger.info(f"Starting invoice field mapping from {input_dir} to {output_dir}")
    process_all_files(input_dir)
    logger.info("Invoice field mapping completed")

if __name__ == "__main__":
    main()

2025-09-10 13:36:36,305 - INFO - Starting invoice field mapping from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped
2025-09-10 13:36:36,306 - INFO - Starting field mapping for C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9_processed.txt
2025-09-10 13:36:36,308 - INFO - Extracting key-value pairs from text C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\4. post-processed\9551f2fd-8103-489e-9c6c-e9aefe035ad9_processed.txt
2025-09-10 13:36:37,653 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\dell\miniconda3\envs\ocr_invoice_pipeline\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\dell\miniconda3\envs\ocr_invoice_pipeline\lib\encodings\cp1252.py", li

Structuring Data Into Json

In [ ]:
import os
import glob
import logging
import json

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("invoice_data_structuring.log"), logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def load_text(text_path):
    """Load the mapped text file."""
    with open(text_path, 'r', encoding='utf-8') as f:
        return f.read()

def structure_to_json(text_path):
    """Turn mapped.txt into clean JSON without LLM."""
    logger.info(f"Structuring data from text {text_path}")
    try:
        raw_text = load_text(text_path)
        lines = [line.strip() for line in raw_text.split("\n") if line.strip()]

        structured_data = {"invoice": {"header": {}, "additional_info": {}, "line_items": []}}

        current_section = None
        current_item = {}

        for line in lines:
            # Detect section headers
            if line.startswith("=== "):
                section = line.replace("=", "").strip().lower()
                if "header" in section:
                    current_section = "header"
                elif "additional" in section:
                    current_section = "additional_info"
                elif "line item" in section:
                    current_section = "line_items"
                continue

            # Process key-value pairs
            if ":" in line:
                key, value = [p.strip() for p in line.split(":", 1)]

                if current_section == "header":
                    structured_data["invoice"]["header"][key.lower().replace(" ", "_")] = value
                elif current_section == "additional_info":
                    structured_data["invoice"]["additional_info"][key] = value
                elif current_section == "line_items":
                    current_item[key.lower().replace(" ", "_")] = value

                    # Save when we have a full line item
                    if key.lower().startswith("tax rate"):
                        structured_data["invoice"]["line_items"].append(current_item)
                        current_item = {}

        # Add last line item if not empty
        if current_item:
            structured_data["invoice"]["line_items"].append(current_item)

        return json.dumps(structured_data, indent=2, ensure_ascii=False)

    except Exception as e:
        logger.error(f"Error structuring data: {str(e)}")
        return None

def save_json_data(json_content, input_path, output_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\6. validated json"):
    """Save the structured data to a JSON file."""
    logger.info(f"Preparing to save JSON data to {output_dir}")
    try:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(input_path).replace("_mapped.txt", "_mapped.json")
        json_output_path = os.path.join(output_dir, base_name)
        with open(json_output_path, 'w', encoding='utf-8') as f:
            f.write(json_content)
        logger.info(f"JSON data saved to {json_output_path}")
        return json_output_path
    except Exception as e:
        logger.error(f"Error saving JSON data: {e}")
        raise

def process_all_files(input_dir=r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped"):
    """Process all mapped text files in the input directory."""
    if not os.path.exists(input_dir):
        logger.error(f"Input directory {input_dir} does not exist")
        return
    mapped_files = glob.glob(os.path.join(input_dir, "*_mapped.txt"))
    if not mapped_files:
        logger.error(f"No mapped text files found in {input_dir}")
        return
    for mapped_file in mapped_files:
        logger.info(f"Processing mapped file: {mapped_file}")
        json_content = structure_to_json(mapped_file)
        if json_content:
            save_json_data(json_content, mapped_file)
        else:
            logger.warning(f"No valid JSON data for {mapped_file}")

def main():
    input_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped"
    output_dir = r"C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\6. validated json"
    logger.info(f"Starting invoice data structuring from {input_dir} to {output_dir}")
    process_all_files(input_dir)
    logger.info("Invoice data structuring completed")

if __name__ == "__main__":
    main()


2025-09-10 13:37:28,324 - INFO - Starting invoice data structuring from C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\6. validated json
2025-09-10 13:37:28,326 - INFO - Processing mapped file: C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped\9551f2fd-8103-489e-9c6c-e9aefe035ad9_mapped.txt
2025-09-10 13:37:28,326 - INFO - Structuring data from text C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\5. mapped\9551f2fd-8103-489e-9c6c-e9aefe035ad9_mapped.txt
2025-09-10 13:37:28,328 - INFO - Preparing to save JSON data to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\6. validated json
2025-09-10 13:37:28,335 - INFO - JSON data saved to C:\Users\dell\OneDrive\Desktop\OCR_Project\ocr_facture13\6. validated json\9551f2fd-8103-489e-9c6c-e9aefe035ad9_mapped.json
2025-09-10 13:37:28,336 - INFO - Invoice data structuring completed
